## SAM Segmentation for comparison

In [20]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config as cfg
import cv2
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from segment_anything import sam_model_registry, SamPredictor
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

sam = sam_model_registry["vit_b"](checkpoint=cfg.SAM_CHECKPOINT_PATH).to(device)
predictor = SamPredictor(sam)

Using device: cuda


c:\venv_codeskr\torch_cuda_env\Lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


## Sample original images

In [25]:
"""
Sample N images (stratified across variety/class folders) for manual segmentation.

Copies each picked ORIGINAL image into:
    Manual Segmentation/Original image/<variety>/<class>/<file>

so an annotator can draw ground-truth masks on the originals.

Reproducible: fixed SEED + deterministic file listing. Writes a manifest.csv
documenting exactly what was sampled (for the reviewer response / provenance).
"""

import csv
import random
import shutil
from pathlib import Path

# natsort optional — graceful fallback to sorted()
try:
    from natsort import natsorted as _sort
except Exception:
    _sort = sorted

# ----------------------------- config -----------------------------
DATA_DIR   = Path("Data")                 # contains "Original image"
ORIG_NAME  = "Original image"
OUT_DIR    = Path("Manual Segmentation")
N_TOTAL    = 60
SEED       = 42
IMG_EXTS   = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}
# ------------------------------------------------------------------

orig_root = DATA_DIR / ORIG_NAME


def list_images(folder: Path):
    return _sort([p.name for p in folder.iterdir()
                  if p.is_file() and p.suffix.lower() in IMG_EXTS])


def find_strata(root: Path):
    """Return {relative_folder: [filenames]} for every leaf folder holding images."""
    strata = {}
    for p in _sort([str(x) for x in root.rglob("*")]):
        p = Path(p)
        if p.is_dir():
            imgs = list_images(p)
            if imgs:
                strata[p.relative_to(root)] = imgs
    return strata


def allocate(counts: dict, total: int):
    """Spread `total` across strata as evenly as possible, capped by availability."""
    alloc = {s: 0 for s in counts}
    remaining = min(total, sum(counts.values()))
    active = [s for s in counts if counts[s] > alloc[s]]
    while remaining > 0 and active:
        share = max(1, remaining // len(active))
        for s in active:
            if remaining <= 0:
                break
            add = min(share, counts[s] - alloc[s], remaining)
            alloc[s] += add
            remaining -= add
        active = [s for s in counts if counts[s] > alloc[s]]
    return alloc


def main():
    rng = random.Random(SEED)
    strata = find_strata(orig_root)
    if not strata:
        raise SystemExit(f"No images found under {orig_root}")

    counts = {s: len(f) for s, f in strata.items()}
    alloc = allocate(counts, N_TOTAL)

    manifest = []
    for rel_folder in _sort([str(s) for s in strata]):
        rel_folder = Path(rel_folder)
        n = alloc[rel_folder]
        if n == 0:
            continue
        picked = rng.sample(strata[rel_folder], n)  # seeded, without replacement
        for fname in _sort(picked):
            src_orig = orig_root / rel_folder / fname
            dst_orig = OUT_DIR / ORIG_NAME / rel_folder / fname
            dst_orig.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_orig, dst_orig)

            manifest.append({
                "stratum": str(rel_folder),
                "filename": fname,
            })

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(OUT_DIR / "manifest.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["stratum", "filename"])
        w.writeheader()
        w.writerows(manifest)

    # ---- report ----
    n_strata = len([s for s in alloc if alloc[s]])
    print(f"Sampled {len(manifest)} images across {n_strata} strata (seed={SEED}):")
    for s in _sort([str(x) for x in strata]):
        if alloc[Path(s)]:
            print(f"  {s:<40} {alloc[Path(s)]:>3}  (of {counts[Path(s)]} available)")
    print(f"\nManifest written to {OUT_DIR/'manifest.csv'}")


if __name__ == "__main__":
    main()

Sampled 60 images across 9 strata (seed=42):
  BPT5204\Broken                             7  (of 1001 available)
  BPT5204\Discolored                         7  (of 4057 available)
  BPT5204\Full_chalky                        7  (of 2399 available)
  BPT5204\Healthy                            7  (of 2277 available)
  BPT5204\Partial_chalky                     7  (of 2366 available)
  BPT5204\Peck_damage                        7  (of 2128 available)
  Basmati and non-basmati\PB1121             6  (of 4994 available)
  Basmati and non-basmati\PB1401             6  (of 2499 available)
  Basmati and non-basmati\PR14               6  (of 2501 available)

Manifest written to Manual Segmentation\manifest.csv


## Generate new segment mask using AM for comparison (no aligning this time)

In [21]:
# ============================================================================
# Regenerate SAM masks in the ORIGINAL image frame (NO align_vertical) for the
# manual-segmentation validation set. Peck grains use sam_mask_peck, everything
# else uses sam_mask. Masks are written binary (white grain on black) so they
# line up pixel-for-pixel with the manual annotations.
#
# Assumes `cfg` and `predictor` (SAM) are already defined earlier in the notebook.
# Delete the old contents of "Manual Segmentation/Segmented image" before running.
# ============================================================================

from tqdm import tqdm
from datetime import datetime
from pathlib import Path
import cv2, numpy as np


# ------------------------- segmentation helpers -----------------------------
def largest_connected_component(mask):
    """Keep only the largest blob in a binary mask."""
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return mask
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    out = np.zeros_like(mask)
    out[labels == largest] = 255
    return out


def pick_grain_component(closed, S, min_area=2000):
    """Choose the most grain-like blob: compact, low-saturation (grey), elongated,
    near the frame centre — so a grey background corner patch cannot win."""
    n, labels, stats, cent = cv2.connectedComponentsWithStats(closed, 8)
    if n <= 1:
        return closed
    H, W = closed.shape
    best, bi = -1.0, 0
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        m = labels == i
        bw, bh = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        cx, cy = cent[i]
        fill   = area / (bw * bh)
        sat    = 1.0 - min(np.median(S[m]) / cfg.HLS_S_MAX, 1.0)
        elong  = max(bw, bh) / max(min(bw, bh), 1)
        elong  = min(elong / 3.0, 1.0)
        centre = 1.0 - min(np.hypot((cx - W/2)/(W/2), (cy - H/2)/(H/2)), 1.0)
        grain_score = fill * sat * elong * centre
        if grain_score > best:
            best, bi = grain_score, i
    out = np.zeros_like(closed)
    out[labels == bi] = 255
    return out


def get_seedpoints(image_bgr, scale=4, erode_frac=0.11, spec_k=2.5,
                   margin_frac=0.02, matte_bias=0.65, n_seeds=2):
    """Positive point prompts for SAM (appearance-aware, for undamaged/regular grains)."""
    h, w = image_bgr.shape[:2]
    small = cv2.resize(image_bgr, (w // scale, h // scale), interpolation=cv2.INTER_AREA)
    hls = cv2.cvtColor(small, cv2.COLOR_BGR2HLS)
    L, S = hls[:, :, 1], hls[:, :, 2]

    raw = cv2.inRange(hls, np.array([0, 0, cfg.HLS_S_MIN], np.uint8),
                           np.array([255, 255, cfg.HLS_S_MAX], np.uint8))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(cv2.morphologyEx(raw, cv2.MORPH_OPEN, k, 2),
                              cv2.MORPH_CLOSE, k, 2)
    ko = max(7, int(min(h, w) / scale * 0.02)) | 1
    closed = cv2.morphologyEx(closed, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ko, ko)))
    loose = pick_grain_component(closed, S, min_area=2000)

    d = dict(small=small, L=L, S=S, raw=raw, closed=closed, loose=loose,
             scale=scale, n_seeds=n_seeds)
    if loose.sum() == 0:
        d.update(core=loose, highlight=loose, allowed=loose,
                 score=loose.astype(np.float32), seeds=[(w // 2, h // 2)], rel_thr=0)
        return d

    diam = 2 * np.sqrt(loose.sum() / 255 / np.pi)
    er = max(2, int(diam * erode_frac))
    core = largest_connected_component(
        cv2.erode(loose, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*er+1, 2*er+1)), 1))
    if core.sum() == 0:
        core = loose.copy()
    cb = core > 0

    Lc, Sc = L[cb], S[cb]
    rel_thr = float(np.median(Lc) + spec_k * Lc.std())
    highlight = ((L >= rel_thr) & (S <= np.median(Sc)) & cb).astype(np.uint8) * 255
    r = max(1, int(diam * margin_frac))
    hl_d = cv2.dilate(highlight, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)))
    allowed = cv2.bitwise_and(core, cv2.bitwise_not(hl_d))
    if allowed.sum() == 0:
        allowed = core.copy()

    dt = cv2.distanceTransform(allowed, cv2.DIST_L2, 5)
    if dt.max() == 0:
        dt = cv2.distanceTransform(core, cv2.DIST_L2, 5)
    dt_n = dt / (dt.max() or 1)
    lo, hi = np.percentile(Lc, 5), np.percentile(Lc, 95)
    Ln = np.clip((L.astype(np.float32) - lo) / max(hi - lo, 1), 0, 1)
    score = dt_n * ((1 - matte_bias) + matte_bias * (1 - Ln))
    score[allowed == 0] = 0

    ys = np.where(cb.max(axis=1))[0]
    y0, y1 = ys.min(), ys.max()
    seeds = []
    for i in range(n_seeds):
        band = score.copy()
        band[:y0 + (y1 - y0) * i // n_seeds] = 0
        band[y0 + (y1 - y0) * (i + 1) // n_seeds:] = 0
        if band.max() > 0:
            sy, sx = np.unravel_index(np.argmax(band), band.shape)
            seeds.append((int(sx * scale), int(sy * scale)))
    if not seeds:
        seeds = [(w // 2, h // 2)]

    d.update(core=core, highlight=highlight, allowed=allowed, score=score,
             seeds=seeds, rel_thr=rel_thr)
    return d


def fill_holes(mask):
    """Fill internal holes in a binary mask (e.g. bran/chalk gaps inside the grain)."""
    h, w = mask.shape[:2]
    ff = mask.copy()
    m2 = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(ff, m2, (0, 0), 255)
    holes = cv2.bitwise_not(ff)
    return cv2.bitwise_or(mask, holes)


def check_segmentation(mask, min_area_frac=0.03, max_patch_frac=0.10):
    """Validate a final grain mask. Returns (ok: bool, reason: str)."""
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return False, "empty"
    areas = stats[1:, cv2.CC_STAT_AREA]
    order = np.argsort(areas)[::-1]
    main = areas[order[0]]
    if main / mask.size < min_area_frac:
        return False, f"too small ({main / mask.size:.4f})"
    if len(areas) > 1 and areas[order[1]] / main > max_patch_frac:
        return False, f"has patches ({len(areas)} blobs)"
    return True, "ok"


def sam_mask(image_bgr):
    """Segment one grain: SAM from seed points, then HLS shadow filter + largest component."""
    predictor.set_image(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    S = get_seedpoints(image_bgr)
    pts = np.array(S["seeds"]);  labels = np.ones(len(pts), int)
    masks, scores, _ = predictor.predict(point_coords=pts, point_labels=labels,
                                         multimask_output=True)
    mask = masks[np.argmax(scores)].astype(np.uint8) * 255
    hls = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HLS)
    mask = cv2.bitwise_and(mask, cv2.inRange(hls, np.array([0, 0, cfg.HLS_S_MIN], np.uint8),
                                                  np.array([255, 255, cfg.HLS_S_MAX], np.uint8)))
    mask = largest_connected_component(mask)
    mask = fill_holes(mask)
    return largest_connected_component(mask)


def get_seedpoints_peck(image_bgr, scale=4, n_seeds=3, reach=0.85):
    """Peck-damage seeds: purely geometric (top/mid/bottom of the loose grain silhouette)."""
    h, w = image_bgr.shape[:2]
    small = cv2.resize(image_bgr, (w // scale, h // scale), interpolation=cv2.INTER_AREA)
    hls = cv2.cvtColor(small, cv2.COLOR_BGR2HLS)
    L, S = hls[:, :, 1], hls[:, :, 2]

    raw = cv2.inRange(hls, np.array([0, 0, cfg.HLS_S_MIN], np.uint8),
                           np.array([255, 255, cfg.HLS_S_MAX], np.uint8))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    closed = cv2.morphologyEx(cv2.morphologyEx(raw, cv2.MORPH_OPEN, k, 2),
                              cv2.MORPH_CLOSE, k, 2)
    ko = max(7, int(min(h, w) / scale * 0.02)) | 1
    closed = cv2.morphologyEx(closed, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ko, ko)))
    loose = pick_grain_component(closed, S, min_area=2000)
    loose = fill_holes(loose)

    d = dict(small=small, L=L, S=S, raw=raw, closed=closed, loose=loose,
             scale=scale, n_seeds=n_seeds)
    if loose.sum() == 0:
        d["seeds"] = [(w // 2, h // 2)]
        return d

    lb = loose > 0
    ys = np.where(lb.max(axis=1))[0]
    y0, y1 = ys.min(), ys.max()
    seeds = []
    for i in range(n_seeds):
        t = (i + 0.5) / n_seeds
        ty = int(round(y0 + (y1 - y0) * (reach * t + (1 - reach) * 0.5)))
        row = ty
        for dy in range(0, (y1 - y0) // 2 + 1):
            if lb[min(max(ty + dy, 0), lb.shape[0]-1)].any():
                row = min(max(ty + dy, 0), lb.shape[0]-1);  break
            if lb[min(max(ty - dy, 0), lb.shape[0]-1)].any():
                row = min(max(ty - dy, 0), lb.shape[0]-1);  break
        xs = np.where(lb[row])[0]
        cx = int((xs.min() + xs.max()) / 2)
        seeds.append((int(cx * scale), int(row * scale)))
    d["seeds"] = seeds
    return d


def sam_mask_peck(image_bgr):
    """Segment a peck-damage grain: geometric seeds, then HLS filter + largest component."""
    predictor.set_image(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    S = get_seedpoints_peck(image_bgr, n_seeds=3)
    pts = np.array(S["seeds"]);  labels = np.ones(len(pts), int)
    masks, scores, _ = predictor.predict(point_coords=pts, point_labels=labels,
                                         multimask_output=True)
    mask = masks[np.argmax(scores)].astype(np.uint8) * 255
    hls = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HLS)
    mask = cv2.bitwise_and(mask, cv2.inRange(hls, np.array([0, 0, cfg.HLS_S_MIN], np.uint8),
                                                  np.array([255, 255, cfg.HLS_S_MAX], np.uint8)))
    mask = largest_connected_component(mask)
    mask = fill_holes(mask)
    return largest_connected_component(mask)


# ------------------------------- batch run ----------------------------------
VALID = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
SRC_DIR  = Path("Manual Segmentation") / "Original image"     # the 60 annotated originals
MASK_DIR = Path("Manual Segmentation") / "Segmented image"    # masks written here
LOG_PATH = MASK_DIR / "regen_failures.log"
PECK_FOLDER_NAMES = {"peck_damage"}                           # folders routed to sam_mask_peck

MASK_DIR.mkdir(parents=True, exist_ok=True)

image_paths = [p for p in SRC_DIR.rglob("*")
               if p.is_file() and p.suffix.lower() in VALID]


def mask_for(image, rel):
    is_peck = bool(PECK_FOLDER_NAMES & {part.lower() for part in rel.parts})
    return (sam_mask_peck(image), "peck") if is_peck else (sam_mask(image), "general")


failures, processed, peck_n = [], 0, 0
log = open(LOG_PATH, "w", buffering=1)
log.write(f"Mask regen (no align): {datetime.now():%Y-%m-%d %H:%M:%S}\n")
log.write(f"Total images: {len(image_paths)}\n" + "-" * 60 + "\n"); log.flush()


def record(rel, why):
    failures.append((rel, why)); log.write(f"[{why}]\t{rel}\n"); log.flush()


try:
    for img_path in tqdm(image_paths, desc="Regen masks (no align)", unit="image"):
        rel = img_path.relative_to(SRC_DIR); processed += 1

        image = cv2.imread(str(img_path))
        if image is None:
            record(rel, "unreadable"); continue

        mask, route = mask_for(image, rel)
        peck_n += (route == "peck")
        if mask is None or mask.sum() == 0:
            record(rel, "empty mask"); continue

        ok, reason = check_segmentation(mask)
        if not ok:
            record(rel, reason); continue

        # NO align_vertical: mask stays in ORIGINAL coordinates, matching the annotation
        out = MASK_DIR / rel.parent / f"{img_path.stem}.png"
        out.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out), (mask > 0).astype(np.uint8) * 255)   # white grain on black

except KeyboardInterrupt:
    print("\nInterrupted -- log saved up to this point.")

finally:
    log.write("-" * 60 + f"\nEnded: {datetime.now():%Y-%m-%d %H:%M:%S}\n")
    log.write(f"Processed {processed}/{len(image_paths)} | peck-routed {peck_n} | "
              f"Failed {len(failures)}\n")
    log.flush(); log.close()

print(f"Processed {processed}/{len(image_paths)}  |  peck-routed {peck_n}  |  "
      f"{len(failures)} failed  ->  {LOG_PATH}")

Regen masks (no align): 100%|██████████| 60/60 [02:16<00:00,  2.27s/image]

Processed 60/60  |  peck-routed 7  |  0 failed  ->  Manual Segmentation\Segmented image\regen_failures.log


## Analysis

Output: IoU and Dice

In [27]:
"""
Quantitative validation of SAM + HLS segmentation against manual annotation.

For each manually annotated image:
  * ground-truth mask  <- rasterized from the Labelme polygon(s)
  * predicted mask     <- binarized from the SAM 'Segmented image' output
  * IoU and Dice computed in the ORIGINAL image coordinate frame

Outputs:
  results_per_image.csv   one row per image (IoU, Dice, areas, flags)
  summary_per_class.csv   per-variety/class + overall mean, SD, 95% CI, n
  GT_masks/...            rasterized ground-truth masks (binary PNG)
  QA_overlays/...         GT(green)/SAM(red)/overlap(yellow) on original (optional)

Boundary convention assumed to match the annotator's rule:
  grain = union of all annotated polygons; background everything else.
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from scipy import stats

# ------------------------------- config -------------------------------
ROOT        = Path("Manual Segmentation")                      # run from the folder holding the dirs below
ANN_DIR     = ROOT / "Labelme_Annotations" / "Annotations"   # *.json live here
SEG_DIR_HINT = "Segmented image"                        # auto-detected: dir name containing this (typo-robust)
ORIG_DIR_HINT = "Original image"                   # auto-detected: dir name containing this
OUT_DIR     = ROOT / "Labelme_Annotations"   # GT_masks / QA_overlays / csv go here

BG_THRESH     = 12        # grayscale distance from bg colour to count as foreground
LABEL_FILTER  = None      # e.g. "grain" to use only that label; None = all polygons
SAVE_OVERLAYS = True      # write QA overlays (needs originals)
LOW_IOU_WARN  = 0.50      # median IoU below this => likely coordinate-frame mismatch
IMG_EXTS      = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}
# ----------------------------------------------------------------------


def autodetect_dir(root: Path, hint: str):
    for p in sorted(root.iterdir()):
        if p.is_dir() and hint.lower() in p.name.lower():
            return p
    return None


def rasterize(shapes, h, w):
    """Union of all polygons/rectangles/circles -> binary mask (uint8 {0,1})."""
    img = Image.new("L", (w, h), 0)
    d = ImageDraw.Draw(img)
    for shp in shapes:
        if LABEL_FILTER and shp.get("label") != LABEL_FILTER:
            continue
        pts = [tuple(map(float, p)) for p in shp["points"]]
        st = shp.get("shape_type", "polygon")
        if st == "polygon" and len(pts) >= 3:
            d.polygon(pts, fill=1)
        elif st == "rectangle" and len(pts) == 2:
            (x0, y0), (x1, y1) = pts
            d.rectangle([min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)], fill=1)
        elif st == "circle" and len(pts) == 2:
            (cx, cy), (px, py) = pts
            r = ((px - cx) ** 2 + (py - cy) ** 2) ** 0.5
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=1)
    return np.asarray(img, dtype=np.uint8)


def binarize_sam(path: Path):
    """SAM foreground -> binary mask. Auto-detects black vs white background."""
    im = np.asarray(Image.open(path).convert("L"))
    corners = [im[0, 0], im[0, -1], im[-1, 0], im[-1, -1]]
    bg = "white" if np.mean(corners) > 127 else "black"
    if bg == "black":
        return (im > BG_THRESH).astype(np.uint8), bg
    return (im < 255 - BG_THRESH).astype(np.uint8), bg


def find_by_stem(folder: Path, stem: str):
    if not folder.is_dir():
        return None
    for p in sorted(folder.iterdir()):
        if p.is_file() and p.stem == stem and p.suffix.lower() in IMG_EXTS:
            return p
    return None


def iou_dice(gt, pred):
    inter = int(np.logical_and(gt, pred).sum())
    gt_a, pr_a = int(gt.sum()), int(pred.sum())
    union = gt_a + pr_a - inter
    iou = inter / union if union else 1.0
    dice = 2 * inter / (gt_a + pr_a) if (gt_a + pr_a) else 1.0
    return iou, dice, gt_a, pr_a


def ci95(x):
    x = np.asarray(x, float)
    n = len(x)
    if n < 2:
        return (np.nan, np.nan)
    m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
    h = stats.t.ppf(0.975, n - 1) * se
    return m - h, m + h


def save_overlay(orig_path, gt, pred, dest):
    base = Image.open(orig_path).convert("RGB").resize((gt.shape[1], gt.shape[0]))
    ov = np.asarray(base, dtype=np.uint8).copy()
    only_gt = (gt == 1) & (pred == 0)
    only_pr = (pred == 1) & (gt == 0)
    both = (gt == 1) & (pred == 1)
    ov[only_gt] = (0.4 * ov[only_gt] + np.array([0, 153, 0])).clip(0, 255)   # green
    ov[only_pr] = (0.4 * ov[only_pr] + np.array([204, 0, 0])).clip(0, 255)   # red
    ov[both]    = (0.4 * ov[both]    + np.array([204, 204, 0])).clip(0, 255)  # yellow
    dest.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(ov.astype(np.uint8)).save(dest)


def main():
    seg_dir = autodetect_dir(ROOT, SEG_DIR_HINT)
    orig_dir = autodetect_dir(ROOT, ORIG_DIR_HINT)
    if seg_dir is None:
        raise SystemExit(f"Could not find a '*{SEG_DIR_HINT}*' folder in {ROOT.resolve()}")
    print(f"SAM segmented dir : {seg_dir}")
    print(f"Original dir      : {orig_dir}")
    print(f"Annotations dir   : {ANN_DIR}\n")

    rows, unmatched, mismatched = [], [], []
    for jpath in sorted(ANN_DIR.rglob("*.json")):
        rel = jpath.relative_to(ANN_DIR).parent          # <variety>/<class>
        stem = jpath.stem
        data = json.loads(jpath.read_text())
        h, w = data["imageHeight"], data["imageWidth"]
        gt = rasterize(data.get("shapes", []), h, w)

        seg_path = find_by_stem(seg_dir / rel, stem)
        if seg_path is None:
            unmatched.append(str(rel / stem))
            continue
        pred, bg = binarize_sam(seg_path)

        resized = False
        if pred.shape != gt.shape:                        # frame/size guard
            resized = True
            mismatched.append(f"{rel/stem}: GT{gt.shape} vs SAM{pred.shape}")
            pred = np.asarray(
                Image.fromarray(pred * 255).resize((w, h), Image.NEAREST)
            ) // 255

        iou, dice, gt_a, pr_a = iou_dice(gt, pred)

        gt_out = OUT_DIR / "GT_masks" / rel / f"{stem}.png"
        gt_out.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(gt * 255).save(gt_out)

        if SAVE_OVERLAYS and orig_dir is not None:
            op = find_by_stem(orig_dir / rel, stem)
            if op is not None:
                save_overlay(op, gt, pred,
                             OUT_DIR / "QA_overlays" / rel / f"{stem}.png")

        parts = rel.parts
        rows.append({
            "variety": parts[0] if len(parts) > 0 else "",
            "class":   parts[1] if len(parts) > 1 else "",
            "filename": stem,
            "IoU": round(iou, 4),
            "Dice": round(dice, 4),
            "gt_px": gt_a,
            "sam_px": pr_a,
            "sam_bg": bg,
            "resized": resized,
        })

    if not rows:
        raise SystemExit("No matched image pairs found — check folder names/structure.")

    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / "results_per_image.csv", index=False)

    # ---- per-class + overall summary ----
    def summarize(g, name):
        lo_i, hi_i = ci95(g["IoU"]); lo_d, hi_d = ci95(g["Dice"])
        return {
            "group": name, "n": len(g),
            "IoU_mean": round(g["IoU"].mean(), 4),
            "IoU_sd": round(g["IoU"].std(ddof=1), 4) if len(g) > 1 else np.nan,
            "IoU_CI95": f"[{lo_i:.4f}, {hi_i:.4f}]" if len(g) > 1 else "n/a",
            "Dice_mean": round(g["Dice"].mean(), 4),
            "Dice_sd": round(g["Dice"].std(ddof=1), 4) if len(g) > 1 else np.nan,
            "Dice_CI95": f"[{lo_d:.4f}, {hi_d:.4f}]" if len(g) > 1 else "n/a",
        }

    summary = [summarize(g, f"{v}/{c}")
               for (v, c), g in df.groupby(["variety", "class"])]
    summary.append(summarize(df, "OVERALL"))
    sdf = pd.DataFrame(summary)
    sdf.to_csv(OUT_DIR / "summary_per_class.csv", index=False)

    # ---- report ----
    pd.set_option("display.width", 140, "display.max_columns", 20)
    print(sdf.to_string(index=False))
    print(f"\nMatched pairs: {len(df)}")
    if unmatched:
        print(f"[!] {len(unmatched)} annotations had no SAM image: {unmatched[:5]}"
              f"{' ...' if len(unmatched) > 5 else ''}")
    if mismatched:
        print(f"[!] {len(mismatched)} size mismatches (pred resized to GT):")
        for m in mismatched[:5]:
            print(f"    {m}")

    med = df["IoU"].median()
    if med < LOW_IOU_WARN:
        print(f"\n*** WARNING: median IoU = {med:.3f} is very low. This usually means "
              "the SAM 'Segmented image' files are NOT in the same coordinate frame as "
              "the originals the annotator drew on (e.g. rotated/cropped by align_vertical). "
              "Compare against SAM masks in the ORIGINAL frame, not the reoriented output.")

    print(f"\nWrote: results_per_image.csv, summary_per_class.csv, GT_masks/"
          + (", QA_overlays/" if SAVE_OVERLAYS else ""))


if __name__ == "__main__":
    main()

SAM segmented dir : Manual Segmentation\Segmented image
Original dir      : Manual Segmentation\Original image
Annotations dir   : Manual Segmentation\Labelme_Annotations\Annotations

                         group  n  IoU_mean  IoU_sd         IoU_CI95  Dice_mean  Dice_sd        Dice_CI95
                BPT5204/Broken  7    0.9829  0.0040 [0.9792, 0.9866]     0.9914   0.0020 [0.9895, 0.9933]
            BPT5204/Discolored  7    0.9877  0.0019 [0.9859, 0.9895]     0.9938   0.0010 [0.9929, 0.9947]
           BPT5204/Full_chalky  7    0.9877  0.0023 [0.9856, 0.9898]     0.9938   0.0012 [0.9927, 0.9949]
               BPT5204/Healthy  7    0.9875  0.0026 [0.9851, 0.9899]     0.9937   0.0013 [0.9925, 0.9950]
        BPT5204/Partial_chalky  7    0.9883  0.0023 [0.9861, 0.9904]     0.9941   0.0012 [0.9930, 0.9952]
           BPT5204/Peck_damage  7    0.9866  0.0022 [0.9846, 0.9886]     0.9933   0.0011 [0.9922, 0.9943]
Basmati and non-basmati/PB1121  6    0.9829  0.0043 [0.9785, 0.9874]     0

In [29]:
import rice_common as rc
from collections import Counter
bad = [(p.parent.name, rc.parse_bpt_acronym(p))
       for p in (cfg.SEGMENTED_IMAGE_DIR/"BPT5204").rglob("*")
       if p.suffix.lower() in rc.VALID_EXT and rc.parse_bpt_acronym(p)
       and rc.ACR_TO_LABEL[rc.parse_bpt_acronym(p)] != rc.BPT_DIR_TO_LABEL.get(p.parent.name)]
print(len(bad), Counter(bad))

0 Counter()
